# ДЗ 1. Банковские счета

Павлычев Артемий, БРАПС, 1 курс. Сделал все 3 задания

Файлы для третьего задания лежат в `data/` рядом с файлом, можно скачать из репозитория: https://github.com/linkartemy/hse-ms-python/tree/main

## Как я понял неоднозначные места

- История - это список объектов `Operation`, в нем есть тип, сумма, дата и время, баланс после операции и статус. `get_history()` - отдает копию в виде словарей
- Неправильная сумма (ноль, минус, не число) - вызывает исключение `InvalidAmountError`. А нехватка денег - не исключение: попытка пишется в историю со статусом `fail`, `withdraw()` возвращает `False`. Судя по условию
- Имя владельца - два слова с большой буквы, целиком латиницей или целиком кириллицей
- Счетчик номеров общий для всех типов счетов: `ACC-1000`, `ACC-1001` и т.д.
- Последние n крупных операций - беру n успешных операций с самой большой суммой, с порогом `min_amount` - n самых свежих операций не меньше порога
- `apply_interest(rate)` принимает ставку в процентах: `7` значит плюс 7% к остатку

## Классы

### Исключения и операция

In [1]:
import csv
import json
import math
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path


class AccountError(Exception):
    """Общая ошибка для операций со счетом"""


class InvalidHolderError(AccountError, ValueError):
    """Имя владельца не в формате Имя Фамилия"""


class InvalidAmountError(AccountError, ValueError):
    """Сумма не положительное число"""


def is_number(value):
    """True для обычного конечного числа. bool сюда не подходит"""
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and math.isfinite(value)
    )


@dataclass(frozen=True)
class Operation:
    """Запись в истории счета"""

    operation_type: str   # deposit, withdraw или interest
    amount: float
    timestamp: datetime
    balance_after: float  # баланс сразу после операции
    status: str           # success или fail

    def to_dict(self):
        return {
            "date": self.timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            "type": self.operation_type,
            "amount": round(self.amount, 2),
            "balance_after": round(self.balance_after, 2),
            "status": self.status,
        }

    def __str__(self):
        return (
            f"{self.timestamp:%Y-%m-%d %H:%M:%S} | "
            f"{self.operation_type:<8} | {self.amount:>10.2f} | "
            f"баланс {self.balance_after:>10.2f} | {self.status}"
        )

### Account

Загрузка из файла (3-е задание) тоже здесь. Для обоих счетов она одинаковая, отличается только список разрешенных операций, каждый класс задает его сам

In [2]:
class Account:

    account_type = "base"
    allowed_operations = ("deposit", "withdraw")

    _account_counter = 1000

    # Два слова с большой буквы: либо латиница, либо кириллица.
    # \u0401 и \u0451 - буква е с двумя точками, в диапазон А-Я
    # она не входит
    _HOLDER_PATTERN = re.compile(
        r"^(?:[A-Z][a-z]+ [A-Z][a-z]+"
        r"|[А-Я\u0401][а-я\u0451]+ [А-Я\u0401][а-я\u0451]+)$"
    )

    _DATE_FORMATS = ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M")
    _VALID_STATUSES = ("success", "fail")

    def __init__(self, account_holder, balance=0):
        self.holder = self._validate_holder(account_holder)
        if not is_number(balance) or balance < 0:
            raise InvalidAmountError(
                f"Начальный баланс не может быть отрицательным, "
                f"получено: {balance!r}"
            )

        # Номер выдаю после проверок
        self.account_number = f"ACC-{Account._account_counter}"
        Account._account_counter += 1
        self._balance = float(balance)
        self.operations_history = []

    def __repr__(self):
        return (
            f"{type(self).__name__}({self.account_number}, "
            f"{self.holder}, баланс {self._balance:.2f})"
        )

    @classmethod
    def _validate_holder(cls, name):
        if not isinstance(name, str) or not cls._HOLDER_PATTERN.match(name):
            raise InvalidHolderError(
                "Имя владельца должно быть в формате Имя Фамилия, "
                f"получено: {name!r}"
            )

        return name

    @staticmethod
    def _validate_amount(amount):
        if not is_number(amount) or amount <= 0:
            raise InvalidAmountError(
                f"Сумма должна быть больше нуля, получено: {amount!r}"
            )

        return float(amount)

    def _record(self, operation_type, amount, status, timestamp=None):
        operation = Operation(
            operation_type=operation_type,
            amount=amount,
            timestamp=timestamp or datetime.now(),
            balance_after=self._balance,
            status=status,
        )
        self.operations_history.append(operation)

        return operation

    def deposit(self, amount):
        amount = self._validate_amount(amount)

        self._balance = round(self._balance + amount, 2)
        self._record("deposit", amount, "success")

    def withdraw(self, amount):
        amount = self._validate_amount(amount)

        if amount > self._balance:
            self._record("withdraw", amount, "fail")
            return False

        self._balance = round(self._balance - amount, 2)
        self._record("withdraw", amount, "success")

        return True

    @property
    def balance(self):
        return self._balance

    def get_balance(self):
        return self._balance

    def get_history(self):
        return [operation.to_dict() for operation in self.operations_history]

    def print_history(self):
        print(f"История {self.account_number} ({self.holder}):")
        for operation in self.operations_history:
            print("  ", operation)

    def get_large_operations(self, n=3, min_amount=None):
        if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
            raise ValueError(
                f"n должно быть целым числом больше нуля, получено: {n!r}"
            )

        successful = [
            op for op in self.operations_history if op.status == "success"
        ]

        if min_amount is None:
            selected = sorted(
                successful,
                key=lambda op: (op.amount, op.timestamp),
                reverse=True,
            )[:n]
        else:
            selected = sorted(
                (op for op in successful if op.amount >= min_amount),
                key=lambda op: op.timestamp,
                reverse=True,
            )[:n]

        selected.sort(key=lambda op: op.timestamp, reverse=True)
        return [op.to_dict() for op in selected]

    # Дальше задание 3: загрузка истории из файла

    def _parse_date(self, raw_date):
        for date_format in self._DATE_FORMATS:
            try:
                return datetime.strptime(str(raw_date).strip(), date_format)
            except ValueError:
                continue

        raise ValueError(f"неправильная дата: {raw_date!r}")

    def _parse_record(self, raw):
        if raw.get("account_type") != self.account_type:
            raise ValueError(
                f"тип счета {raw.get('account_type')!r}, "
                f"а нужен {self.account_type!r}"
            )

        operation_type = (raw.get("operation") or "").strip()
        if not operation_type:
            raise ValueError("пустой тип операции")

        if operation_type not in self.allowed_operations:
            raise ValueError(
                f"операция {operation_type!r} не подходит "
                f"для счета {self.account_type!r}"
            )

        status = (raw.get("status") or "").strip()
        if status not in self._VALID_STATUSES:
            raise ValueError(f"непонятный статус: {status!r}")

        return {
            "type": operation_type,
            "amount": self._parse_amount(raw.get("amount")),
            "date": self._parse_date(raw.get("date")),
            "status": status,
        }

    @staticmethod
    def read_transactions(path):
        path = Path(path)

        if path.suffix == ".csv":
            with open(path, encoding="utf-8", newline="") as file:
                return list(csv.DictReader(file))

        if path.suffix == ".json":
            with open(path, encoding="utf-8") as file:
                return json.load(file)

        raise ValueError(f"Не умею читать файлы {path.suffix}")

    def _parse_amount(self, raw_amount):
        if raw_amount is None or str(raw_amount).strip() == "":
            raise ValueError("пустая сумма")

        try:
            amount = float(raw_amount)
        except ValueError:
            raise ValueError(f"сумма не число: {raw_amount!r}") from None

        if not math.isfinite(amount) or amount <= 0:
            raise ValueError(f"сумма меньше или равна нулю: {amount}")

        return amount

    def load_history(self, path, source_account=None):
        source_account = source_account or self.account_number
        records = [
            record for record in self.read_transactions(path)
            if record.get("account_number") == source_account
        ]
        valid, rejected = self.clean_history(records)

        for record in sorted(valid, key=lambda r: r["date"]):
            amount, status = record["amount"], record["status"]
            if status == "success":
                if record["type"] == "withdraw":
                    if amount > self._balance:
                        rejected.append({
                            "record": record,
                            "reason": "сняли больше, чем было на счете",
                        })
                        continue
                    self._balance = round(self._balance - amount, 2)
                else:  # deposit или interest
                    self._balance = round(self._balance + amount, 2)
            self._record(record["type"], amount, status, record["date"])

        return rejected

    def clean_history(self, records):
        """Проверяет записи из файла до того, как они попадут в историю.
        Выкидывает дубли, опечатки и пустые значения в операции и
        статусе, операции, которых нет у этого типа счета, пустые и
        неположительные суммы, несуществующие даты
        """
        valid, rejected = [], []
        seen = set()

        for raw in records:
            key = tuple(sorted((k, str(v)) for k, v in raw.items()))
            try:
                if key in seen:
                    raise ValueError("дубль")
                seen.add(key)
                valid.append(self._parse_record(raw))
            except ValueError as error:
                rejected.append({"record": raw, "reason": str(error)})

        return valid, rejected

### CheckingAccount и SavingsAccount

In [3]:
class CheckingAccount(Account):  # Расчетный счет
    account_type = "checking"


class SavingsAccount(Account):  # Сберегательный счет
    account_type = "savings"
    allowed_operations = ("deposit", "withdraw", "interest")

    WITHDRAW_LIMIT = 0.5  # за раз можно снять не больше половины баланса

    def withdraw(self, amount):
        amount = self._validate_amount(amount)
        if amount > self._balance * self.WITHDRAW_LIMIT:
            self._record("withdraw", amount, "fail")
            return False

        return super().withdraw(amount)

    def apply_interest(self, rate):
        if not is_number(rate) or rate <= 0:
            raise InvalidAmountError(
                f"Ставка должна быть больше нуля, получено: {rate!r}"
            )

        interest = round(self._balance * rate / 100, 2)
        self._balance = round(self._balance + interest, 2)
        self._record("interest", interest, "success")

        return interest

## Задание 1

In [4]:
account = Account("Иван Петров", 1000)
print(account)

account.deposit(500)
print("Сняли 300:", account.withdraw(300))
print("Сняли 5000, денег не хватает:", account.withdraw(5000))
print("Баланс:", account.get_balance())

# На неправильную сумму - исключение, баланс не меняется
for bad_amount in (-100, 0, "сто"):
    try:
        account.deposit(bad_amount)
    except InvalidAmountError as error:
        print("Ошибка:", error)

account.print_history()

Account(ACC-1000, Иван Петров, баланс 1000.00)
Сняли 300: True
Сняли 5000, денег не хватает: False
Баланс: 1200.0
Ошибка: Сумма должна быть больше нуля, получено: -100
Ошибка: Сумма должна быть больше нуля, получено: 0
Ошибка: Сумма должна быть больше нуля, получено: 'сто'
История ACC-1000 (Иван Петров):
   2026-10-04 19:16:16 | deposit  |     500.00 | баланс    1500.00 | success
   2026-10-04 19:16:16 | withdraw |     300.00 | баланс    1200.00 | success
   2026-10-04 19:16:16 | withdraw |    5000.00 | баланс    1200.00 | fail


In [5]:
for operation in account.get_history():
    print(operation)

# У следующего счета следующий номер
second = Account("John Smith")
print("\nНомер второго счета:", second.account_number)

{'date': '2026-10-04 19:16:16', 'type': 'deposit', 'amount': 500.0, 'balance_after': 1500.0, 'status': 'success'}
{'date': '2026-10-04 19:16:16', 'type': 'withdraw', 'amount': 300.0, 'balance_after': 1200.0, 'status': 'success'}
{'date': '2026-10-04 19:16:16', 'type': 'withdraw', 'amount': 5000.0, 'balance_after': 1200.0, 'status': 'fail'}

Номер второго счета: ACC-1001


## Задание 2

In [6]:
checking = CheckingAccount("Анна Смирнова", 2000)
savings = SavingsAccount("Mary Brown", 10000)
print(checking, checking.account_type)
print(savings, savings.account_type)

# Со сберегательного больше половины баланса за раз не снять
print("\nСнять 6000 из 10000:", savings.withdraw(6000))
print("Снять 3000 из 10000:", savings.withdraw(3000))
print("Начислили 7%:", savings.apply_interest(7))
print("Баланс:", savings.get_balance())

# С расчетного можно снять все
print("\nСнять 2000 из 2000 с расчетного:", checking.withdraw(2000))
savings.print_history()

CheckingAccount(ACC-1002, Анна Смирнова, баланс 2000.00) checking
SavingsAccount(ACC-1003, Mary Brown, баланс 10000.00) savings

Снять 6000 из 10000: False
Снять 3000 из 10000: True
Начислили 7%: 490.0
Баланс: 7490.0

Снять 2000 из 2000 с расчетного: True
История ACC-1003 (Mary Brown):
   2026-10-04 19:16:16 | withdraw |    6000.00 | баланс   10000.00 | fail
   2026-10-04 19:16:16 | withdraw |    3000.00 | баланс    7000.00 | success
   2026-10-04 19:16:16 | interest |     490.00 | баланс    7490.00 | success


In [7]:
bad_names = [
    "иван петров",    # с маленькой буквы
    "Ivan",           # нет фамилии
    "Иван Petrov",    # два алфавита
    "Ivan Petrov 2",  # лишнее слово
    42,               # не строка
]
for name in bad_names:
    try:
        SavingsAccount(name)
    except InvalidHolderError as error:
        print("Ошибка:", error)

try:
    CheckingAccount("Петр Иванов", -50)
except InvalidAmountError as error:
    print("Ошибка:", error)
try:
    checking.deposit(-10)
except InvalidAmountError as error:
    print("Ошибка:", error)

Ошибка: Имя владельца должно быть в формате Имя Фамилия, получено: 'иван петров'
Ошибка: Имя владельца должно быть в формате Имя Фамилия, получено: 'Ivan'
Ошибка: Имя владельца должно быть в формате Имя Фамилия, получено: 'Иван Petrov'
Ошибка: Имя владельца должно быть в формате Имя Фамилия, получено: 'Ivan Petrov 2'
Ошибка: Имя владельца должно быть в формате Имя Фамилия, получено: 42
Ошибка: Начальный баланс не может быть отрицательным, получено: -50
Ошибка: Сумма должна быть больше нуля, получено: -10


In [8]:
analytics = CheckingAccount("Иван Иванов", 0)
for amount in (100, 5000, 250, 7000, 300):
    analytics.deposit(amount)
analytics.withdraw(4000)

print("Три самые крупные операции:")
for operation in analytics.get_large_operations(3):
    print(operation)

print("\nДве последние операции от 1000:")
for operation in analytics.get_large_operations(2, min_amount=1000):
    print(operation)

Три самые крупные операции:
{'date': '2026-10-04 19:16:16', 'type': 'withdraw', 'amount': 4000.0, 'balance_after': 8650.0, 'status': 'success'}
{'date': '2026-10-04 19:16:16', 'type': 'deposit', 'amount': 7000.0, 'balance_after': 12350.0, 'status': 'success'}
{'date': '2026-10-04 19:16:16', 'type': 'deposit', 'amount': 5000.0, 'balance_after': 5100.0, 'status': 'success'}

Две последние операции от 1000:
{'date': '2026-10-04 19:16:16', 'type': 'withdraw', 'amount': 4000.0, 'balance_after': 8650.0, 'status': 'success'}
{'date': '2026-10-04 19:16:16', 'type': 'deposit', 'amount': 7000.0, 'balance_after': 12350.0, 'status': 'success'}


## Задание 3

Файл общий для всех счетов. Номера в нем вида `ACC-100001`, а счетчик по условию выдает `ACC-1000`. Поэтому в `load_history()` передается номер счета из файла, чьи записи нужно загрузить. Если у записи другой тип счета, она отклоняется.

`clean_history()` выкидывает дубли, опечатки и пустые значения в операции и статусе (`diposit`, `wdraw`, `succes`), проценты на расчетном счете, пустые и отрицательные суммы и несуществующие даты вроде `2025-17-34`. Дату `28/09/2025 22:17` ошибкой не считаю, это просто другой формат.

Баланс пересчитываю по самим операциям, колонку `balance_after` не использую: она посчитана вместе с ошибочными строками, и после чистки цифры перестают сходиться. Записи применяются по порядку дат. Лимит 50% у сберегательного счета на загрузку не действует больше

Начальный баланс в примерах взял из самой выгрузки

In [9]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path("data")
# Отсюда файлы скачиваются, если их нет рядом (например, в коллабе)
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "linkartemy/hse-ms-python/main/hw1/data/"
)


def get_data_file(name):
    path = DATA_DIR / name
    if not path.exists():
        DATA_DIR.mkdir(exist_ok=True)
        urlretrieve(DATA_URL + name, path)
    return path


CSV_FILE = get_data_file("transactions_dirty.csv")
JSON_FILE = get_data_file("transactions_dirty.json")
print(CSV_FILE, JSON_FILE)

data/transactions_dirty.csv data/transactions_dirty.json


In [10]:
def show_load(account, path, source_account):
    rejected = account.load_history(path, source_account)
    print(f"{account}: записи {source_account} из {path.name}")
    print(f"загружено {len(account.operations_history)}, "
          f"отклонено {len(rejected)}")
    for item in rejected:
        record = item["record"]
        print(f"  {record.get('date')} {record.get('operation')} "
              f"{record.get('amount')} - {item['reason']}")


checking_loaded = CheckingAccount("Иван Петров", 1200)
show_load(checking_loaded, CSV_FILE, "ACC-100001")
print()
savings_loaded = SavingsAccount("Mary Brown", 1500)
show_load(savings_loaded, JSON_FILE, "ACC-100002")

CheckingAccount(ACC-1005, Иван Петров, баланс 4864.00): записи ACC-100001 из transactions_dirty.csv
загружено 17, отклонено 9
  2025-09-29 22:17:26 withdraw  - пустая сумма
  2025-10-01 22:17:26  654.0 - пустой тип операции
  2025-10-03 22:17:26 interest 532.0 - операция 'interest' не подходит для счета 'checking'
  2025-17-34 12:00:00 deposit 793.0 - неправильная дата: '2025-17-34 12:00:00'
  2025-10-06 22:17:26 diposit 669.0 - операция 'diposit' не подходит для счета 'checking'
  2025-10-08 22:17:26 interest 744.0 - операция 'interest' не подходит для счета 'checking'
  2023-16-40 12:00:00 withdraw 111.0 - неправильная дата: '2023-16-40 12:00:00'
  2025-10-19 22:17:26 deposit 916.0 - непонятный статус: 'succes'
  2025-10-16 22:17:26 withdraw 393.0 - дубль

SavingsAccount(ACC-1006, Mary Brown, баланс 3991.24): записи ACC-100002 из transactions_dirty.json
загружено 18, отклонено 9
  2023-22-32 12:00:00 deposit 797.0 - неправильная дата: '2023-22-32 12:00:00'
  2025-09-30 22:17:26 inter

In [11]:
savings_loaded.print_history()
print("\nБаланс после загрузки:", savings_loaded.get_balance())

История ACC-1006 (Mary Brown):
   2025-09-27 22:17:26 | deposit  |     469.00 | баланс    1969.00 | success
   2025-09-27 22:17:26 | withdraw |     392.00 | баланс    1577.00 | success
   2025-09-28 22:17:26 | deposit  |     317.00 | баланс    1894.00 | success
   2025-10-02 22:17:26 | deposit  |     840.00 | баланс    2734.00 | success
   2025-10-04 22:17:00 | interest |     183.28 | баланс    2917.28 | success
   2025-10-06 22:17:26 | interest |     192.44 | баланс    3109.72 | success
   2025-10-08 22:17:26 | deposit  |     426.00 | баланс    3535.72 | success
   2025-10-09 22:17:00 | deposit  |     638.00 | баланс    4173.72 | success
   2025-10-09 22:17:26 | withdraw |     170.00 | баланс    4003.72 | success
   2025-10-09 22:17:26 | deposit  |     622.00 | баланс    4625.72 | success
   2025-10-10 22:17:26 | interest |     306.26 | баланс    4931.98 | success
   2025-10-12 22:17:26 | withdraw |     278.00 | баланс    4653.98 | success
   2025-10-13 22:17:26 | withdraw |     440.0

In [12]:
# В csv и json одни и те же данные, результат должен совпасть
cases = [
    (CheckingAccount, "ACC-100001", 1200),
    (SavingsAccount, "ACC-100002", 1500),
    (CheckingAccount, "ACC-100003", 800),
    (SavingsAccount, "ACC-100004", 2000),
]
print(f"{'счет':<11} {'тип':<9} {'загружено':>9} {'отклонено':>9} "
      f"{'баланс':>10}  csv = json")
for account_class, source, opening_balance in cases:
    results = []
    for path in (CSV_FILE, JSON_FILE):
        account = account_class("Анна Смирнова", opening_balance)
        rejected = account.load_history(path, source)
        results.append((len(account.operations_history),
                        len(rejected), account.get_balance()))
    loaded, rejected_count, balance = results[0]
    print(f"{source:<11} {account_class.account_type:<9} "
          f"{loaded:>9} {rejected_count:>9} {balance:>10.2f}  "
          f"{results[0] == results[1]}")

счет        тип       загружено отклонено     баланс  csv = json
ACC-100001  checking         17         9    4864.00  True
ACC-100002  savings          18         9    3991.24  True
ACC-100003  checking         13         8    5446.00  True
ACC-100004  savings          19        12   10057.25  True


## Проверки

Если все в порядке, ячейка печатает одну строку. Если что-то сломано, падает с `AssertionError`.

In [13]:
# Задание 1
acc = Account("Test User", 100)
assert acc.account_number.startswith("ACC-")
acc.deposit(50)
assert acc.get_balance() == 150
assert acc.withdraw(200) is False and acc.get_balance() == 150
assert acc.get_history()[-1]["status"] == "fail"
assert acc.withdraw(100) is True and acc.get_balance() == 50
assert set(acc.get_history()[0]) == {
    "date", "type", "amount", "balance_after", "status"}
for bad in (-1, 0, "10", True, float("nan")):
    try:
        acc.deposit(bad)
        raise AssertionError(f"deposit({bad!r}) должен падать")
    except InvalidAmountError:
        pass
next_acc = Account("Test User")
assert int(next_acc.account_number[4:]) == \
    int(acc.account_number[4:]) + 1

# Задание 2
sav = SavingsAccount("Test User", 1000)
assert sav.withdraw(501) is False and sav.withdraw(500) is True
assert sav.apply_interest(10) == 50 and sav.get_balance() == 550
assert issubclass(SavingsAccount, Account)
assert issubclass(CheckingAccount, Account)
assert (CheckingAccount.account_type, SavingsAccount.account_type) \
    == ("checking", "savings")
for bad_name in ("test user", "Test", "Тест User", ""):
    try:
        CheckingAccount(bad_name)
        raise AssertionError(f"имя {bad_name!r} должно падать")
    except InvalidHolderError:
        pass

# Задание 3: все загруженное прошло проверки
for account_class, source, opening_balance in cases:
    loaded = account_class("Test User", opening_balance)
    loaded.load_history(CSV_FILE, source)
    for op in loaded.operations_history:
        assert op.operation_type in account_class.allowed_operations
        assert op.amount > 0 and op.status in ("success", "fail")
    dates = [op.timestamp for op in loaded.operations_history]
    assert dates == sorted(dates)

print("Все проверки прошли")

Все проверки прошли
